# Loop Engineering for Agentic AI — Wanderbricks Demo

**Designed for Databricks Free Edition and a 10–15 minute live webinar demo.**

This notebook uses the built-in `samples.wanderbricks` dataset. The scenario is a **travel support investigation agent** that must investigate a booking, gather evidence from several enterprise tables, decide whether it has enough evidence, produce an answer, and pass a quality gate.

## Notebook flow

```text
User question
    ↓
Initialize state
    ↓
Controller chooses next action
    ↓
Query a Wanderbricks tool/table
    ↓
Observe result + store evidence
    ↓
Evidence quality gate
    ├── Not enough → Reflect / choose another tool → LOOP BACK
    └── Enough
          ↓
      Generate answer
          ↓
      Answer quality gate
          ├── RETRY → revise answer → LOOP BACK
          ├── PASS → return answer
          └── repeated failure → HUMAN REVIEW
```

### What makes this "loop engineering"?

The LLM is **inside** the system, but the loop is engineered around it. The notebook explicitly controls state and memory, tool usage, maximum iterations, evidence sufficiency, answer evaluation, retry limits, and human escalation.

### Free Edition design choices

- Uses serverless notebook compute.
- Uses the preloaded Wanderbricks sample data; no upload is required.
- Does not require Vector Search, Lakebase, LangGraph, or a custom serving endpoint.
- Tries a Databricks-provided model through `ai_query`.
- If the model is unavailable in your Free Edition workspace, the notebook automatically falls back to deterministic demo mode so the live webinar can still run.


In [0]:
# This cell imports the built-in libraries used by the loop and checks that the Wanderbricks sample schema is available.
from pyspark.sql import functions as F
from pyspark.sql import Row
from decimal import Decimal
from datetime import date, datetime
import json

print("Tables available in samples.wanderbricks:")
display(spark.sql("SHOW TABLES IN samples.wanderbricks"))


In [0]:
# This cell loads the small set of Wanderbricks tables used as tools by the support investigation agent.
bookings_df = spark.read.table("samples.wanderbricks.bookings")
payments_df = spark.read.table("samples.wanderbricks.payments")
booking_updates_df = spark.read.table("samples.wanderbricks.booking_updates")
support_df = spark.read.table("samples.wanderbricks.customer_support_logs")
users_df = spark.read.table("samples.wanderbricks.users")

print("BOOKING STATUSES")
display(bookings_df.groupBy("status").count().orderBy(F.desc("count")))

print("PAYMENT STATUSES")
display(payments_df.groupBy("status").count().orderBy(F.desc("count")))

print("SUPPORT LOG SAMPLE")
display(support_df.limit(3))


## Step 1 — Pick a real investigation case

For the webinar we want the loop to investigate a case that has evidence in more than one source.

The notebook first tries to find a **cancelled booking**, with a payment that does **not already look refunded/failed/cancelled**, for a user who also has a support log. If no such record is available, it falls back to any booking with both a payment and a support history.

This makes the demo data-driven rather than hard-coded.


In [0]:
# This cell automatically selects a real Wanderbricks booking that is suitable for demonstrating a multi-step investigation loop.
support_users = support_df.select("user_id").distinct()

booking_payment = (
    bookings_df.alias("b")
    .join(payments_df.alias("p"), F.col("b.booking_id") == F.col("p.booking_id"), "inner")
    .select(
        F.col("b.booking_id").alias("booking_id"),
        F.col("b.user_id").alias("user_id"),
        F.col("b.status").alias("booking_status"),
        F.col("b.total_amount").alias("booking_amount"),
        F.col("p.status").alias("payment_status"),
        F.col("p.amount").alias("payment_amount")
    )
    .join(support_users, on="user_id", how="inner")
)

preferred_case = (
    booking_payment
    .filter(F.lower(F.col("booking_status")).contains("cancel"))
    .filter(~F.lower(F.col("payment_status")).contains("refund"))
    .filter(~F.lower(F.col("payment_status")).contains("fail"))
    .filter(~F.lower(F.col("payment_status")).contains("cancel"))
    .limit(1)
)

candidate = preferred_case.first()
if candidate is None:
    candidate = booking_payment.limit(1).first()
if candidate is None:
    raise RuntimeError("No booking/payment/support combination was found.")

case = candidate.asDict()
user_row = users_df.filter(F.col("user_id") == case["user_id"]).select("user_id", "name").limit(1).first()
customer_name = user_row["name"] if user_row else str(case["user_id"])

QUESTION = (
    f"Investigate booking {case['booking_id']} for customer {customer_name}. "
    f"The booking status is {case['booking_status']} while the payment status is {case['payment_status']}. "
    f"Explain what the available evidence shows and recommend the safest next support action."
)

print("DEMO CASE")
print(json.dumps(case, indent=2, default=str))
print("\nUSER QUESTION")
print(QUESTION)


## Step 2 — Engineer the loop

The demo uses four **tools** backed by Wanderbricks tables:

1. `booking` — current booking record
2. `payment` — payment evidence
3. `booking_updates` — history of booking state changes
4. `support` — customer support messages

The controller does **not** query everything in one SQL join. Instead it keeps a state object, performs one action, observes the result, updates memory, evaluates whether evidence is sufficient, and then decides whether to continue.

```text
PLAN → ACT → OBSERVE → EVALUATE → CONTINUE / ANSWER / ESCALATE
                    ↑                         |
                    └──────── LOOP ───────────┘
```

After evidence collection, a second loop performs:

```text
GENERATE ANSWER → JUDGE → PASS
                     └→ RETRY → GENERATE AGAIN
```


In [0]:
# This cell defines helper functions that turn Spark rows into compact JSON-safe observations for the loop state.
def json_safe(value):
    if isinstance(value, (datetime, date)):
        return value.isoformat()
    if isinstance(value, Decimal):
        return float(value)
    if isinstance(value, Row):
        return {k: json_safe(v) for k, v in value.asDict(recursive=True).items()}
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    return value

def collect_small(df, limit=10):
    return [json_safe(r) for r in df.limit(limit).collect()]

def compact_json(obj, max_chars=7000):
    text = json.dumps(obj, indent=2, default=str)
    return text[:max_chars]


In [0]:
# This cell defines the four Databricks data tools that the loop can call one at a time.
def tool_booking(state):
    return collect_small(bookings_df.filter(F.col("booking_id") == state["booking_id"]), 5)

def tool_payment(state):
    df = payments_df.filter(F.col("booking_id") == state["booking_id"])
    if "payment_date" in payments_df.columns:
        df = df.orderBy(F.desc("payment_date"))
    return collect_small(df, 10)

def tool_booking_updates(state):
    df = booking_updates_df.filter(F.col("booking_id") == state["booking_id"])
    for c in ["booking_update_id", "updated_at"]:
        if c in booking_updates_df.columns:
            df = df.orderBy(F.col(c))
            break
    return collect_small(df, 10)

def tool_support(state):
    user_support = support_df.filter(F.col("user_id") == state["user_id"])
    if "messages" in user_support.columns:
        base_cols = [F.col("ticket_id")] if "ticket_id" in user_support.columns else []
        exploded = user_support.select(*base_cols, F.explode_outer("messages").alias("m"))
        selected = []
        if "ticket_id" in exploded.columns:
            selected.append(F.col("ticket_id"))
        selected.extend([
            F.col("m.message").alias("message"),
            F.col("m.sender").alias("sender"),
            F.col("m.sentiment").alias("sentiment"),
            F.col("m.timestamp").alias("timestamp")
        ])
        return collect_small(exploded.select(*selected), 12)
    return collect_small(user_support, 5)

TOOLS = {
    "booking": tool_booking,
    "payment": tool_payment,
    "booking_updates": tool_booking_updates,
    "support": tool_support
}
print("Available tools:", list(TOOLS.keys()))


In [0]:
# This cell tests a Databricks-provided foundation model through ai_query and enables a deterministic fallback if Free Edition cannot use that model.
MODEL_NAME = "system.ai.gpt-oss-20b"
USE_LLM = True

def call_llm(prompt):
    safe_prompt = prompt.replace("'", "''")
    safe_model = MODEL_NAME.replace("'", "''")
    row = spark.sql(f"SELECT ai_query('{safe_model}', '{safe_prompt}') AS response").first()
    return str(row["response"])

try:
    model_test = call_llm("Reply with exactly: LOOP DEMO READY")
    print(f"LLM mode enabled with {MODEL_NAME}")
    print("Model test:", model_test[:300])
except Exception as e:
    USE_LLM = False
    print("LLM model is not available in this Free Edition workspace.")
    print("The notebook will continue in deterministic fallback mode.")
    print("Change MODEL_NAME to another system.ai model available in your workspace if desired.")
    print("Short error:", str(e)[:500])


In [0]:
# This cell defines the stateful controller and evidence quality gate that decide whether the loop should continue or move to answer generation.
TOOL_SEQUENCE = ["booking", "payment", "booking_updates", "support"]
MAX_TOOL_ITERATIONS = len(TOOL_SEQUENCE)

def initialize_state():
    return {
        "question": QUESTION,
        "booking_id": case["booking_id"],
        "user_id": case["user_id"],
        "evidence": {},
        "actions_taken": [],
        "history": [],
        "status": "RUNNING",
        "iteration": 0
    }

def choose_next_action(state):
    for action in TOOL_SEQUENCE:
        if action not in state["actions_taken"]:
            return action
    return "answer"

def evidence_quality_gate(state):
    booking_seen = "booking" in state["actions_taken"]
    payment_seen = "payment" in state["actions_taken"]
    context_seen = "booking_updates" in state["actions_taken"] and "support" in state["actions_taken"]
    if not booking_seen:
        return "CONTINUE", "Need the current booking record."
    if not payment_seen:
        return "CONTINUE", "Need payment evidence before explaining the conflict."
    if not context_seen:
        return "CONTINUE", "Need booking history and customer-support context."
    return "ANSWER", "Required evidence sources have been investigated."


In [0]:
# This cell defines answer generation and an answer quality gate, creating a second loop for retrying an insufficient answer.
def fallback_answer(state):
    booking = state["evidence"].get("booking", [])
    payment = state["evidence"].get("payment", [])
    updates = state["evidence"].get("booking_updates", [])
    support = state["evidence"].get("support", [])
    booking_status = booking[0].get("status") if booking else "unknown"
    payment_statuses = [str(x.get("status", "unknown")) for x in payment]
    answer = (
        f"Booking {state['booking_id']} currently has booking status '{booking_status}'. "
        f"The related payment record(s) show status {payment_statuses}. "
    )
    if updates:
        answer += "The booking update history was checked to understand how the booking state changed. "
    if support:
        answer += "The customer's support history was reviewed for additional context. "
    answer += (
        "The safest next action is to route the case to the billing/refund support process for operational confirmation, "
        "rather than promising a refund or outcome that is not explicitly supported by the available data."
    )
    return answer

def generate_answer(state, critique=None):
    if not USE_LLM:
        return fallback_answer(state)
    prompt = f"""
You are a support investigation agent for a travel-booking company.
QUESTION: {state['question']}
EVIDENCE FROM ENTERPRISE TABLES: {compact_json(state['evidence'])}
PREVIOUS QUALITY CRITIQUE: {critique or 'None — first attempt.'}

Write a concise answer for a support analyst.
Rules:
1. State only what is supported by the evidence.
2. Mention booking and payment status.
3. Use booking updates/support messages only when relevant.
4. Do not promise a refund unless the evidence explicitly proves it.
5. Recommend the safest next support action.
6. Keep the answer under 180 words.
"""
    return call_llm(prompt)

def judge_answer(state, answer):
    if not USE_LLM:
        ok = "booking" in state["evidence"] and "payment" in state["evidence"] and len(answer) > 80
        return ("PASS", "Deterministic quality gate: mandatory evidence present.") if ok else ("RETRY", "Mandatory evidence is missing.")
    prompt = f"""
You are a strict quality gate for an enterprise agent.
QUESTION: {state['question']}
EVIDENCE: {compact_json(state['evidence'])}
ANSWER: {answer}

Check grounding, booking/payment status coverage, no invented refund/policy outcome, and a safe next action.
Return exactly one line beginning with PASS or RETRY, followed by a short reason.
"""
    raw = call_llm(prompt).strip()
    decision = "PASS" if raw.upper().startswith("PASS") else "RETRY"
    return decision, raw


In [0]:
# This cell defines the main loop: plan one action, call one tool, observe the result, evaluate evidence, and either loop again or generate an answer.
def run_loop():
    state = initialize_state()

    while state["status"] == "RUNNING":
        if state["iteration"] >= MAX_TOOL_ITERATIONS:
            state["status"] = "HUMAN_REVIEW"
            state["history"].append({
                "iteration": state["iteration"] + 1,
                "action": "STOP",
                "rows_observed": 0,
                "decision": "HUMAN_REVIEW",
                "reason": "Maximum tool iterations reached."
            })
            break

        state["iteration"] += 1
        action = choose_next_action(state)
        if action == "answer":
            break

        observation = TOOLS[action](state)
        state["evidence"][action] = observation
        state["actions_taken"].append(action)
        decision, reason = evidence_quality_gate(state)
        state["history"].append({
            "iteration": state["iteration"],
            "action": action,
            "rows_observed": len(observation),
            "decision": decision,
            "reason": reason
        })
        if decision == "ANSWER":
            break

    if state["status"] == "HUMAN_REVIEW":
        return state

    critique = None
    for answer_attempt in range(1, 3):
        answer = generate_answer(state, critique=critique)
        judge_decision, judge_reason = judge_answer(state, answer)
        state["history"].append({
            "iteration": f"A{answer_attempt}",
            "action": "generate_and_judge",
            "rows_observed": len(state["evidence"]),
            "decision": judge_decision,
            "reason": judge_reason[:500]
        })
        state["answer"] = answer
        state["judge_reason"] = judge_reason
        if judge_decision == "PASS":
            state["status"] = "PASS"
            return state
        critique = judge_reason

    state["status"] = "HUMAN_REVIEW"
    return state


In [0]:
# This cell runs the complete loop and displays the iteration history so the webinar audience can see the loop happening step by step.
result = run_loop()

print("=" * 90)
print("LOOP ENGINEERING EXECUTION")
print("=" * 90)
print("\nQUESTION")
print(result["question"])

history_df = spark.createDataFrame(
    [(str(h["iteration"]), h["action"], int(h["rows_observed"]), h["decision"], h["reason"])
     for h in result["history"]],
    ["iteration", "action", "rows_observed", "decision", "reason"]
)
print("\nITERATION HISTORY")
display(history_df)

print("\nFINAL STATUS:", result["status"])
if "answer" in result:
    print("\nFINAL ANSWER")
    print(result["answer"])
print("\nTOOLS USED:", " → ".join(result["actions_taken"]))


In [0]:
# This cell displays the evidence memory accumulated by the loop, making the agent's state and observations transparent.
for source_name, rows in result["evidence"].items():
    print("\n" + "=" * 90)
    print(f"EVIDENCE: {source_name.upper()}")
    print("=" * 90)
    print(compact_json(rows, max_chars=5000))


In [0]:
# This optional cell logs the loop result to MLflow so you can connect the demo to observability and production engineering concepts.
try:
    import mlflow
    with mlflow.start_run(run_name="wanderbricks-loop-engineering-demo"):
        mlflow.log_param("booking_id", str(result["booking_id"]))
        mlflow.log_param("final_status", result["status"])
        mlflow.log_param("tools_used", " -> ".join(result["actions_taken"]))
        mlflow.log_metric("tool_iterations", len(result["actions_taken"]))
        answer_attempts = sum(1 for h in result["history"] if h["action"] == "generate_and_judge")
        mlflow.log_metric("answer_attempts", answer_attempts)
        mlflow.log_dict({
            "question": result["question"],
            "history": result["history"],
            "evidence": result["evidence"],
            "answer": result.get("answer"),
            "judge_reason": result.get("judge_reason")
        }, "loop_execution.json")
    print("MLflow logging completed.")
except Exception as e:
    print("MLflow logging was skipped; the main demo is unaffected.")
    print("Short error:", str(e)[:500])


## Key Observation on the Loop

The **iteration history** matters in this webinar than every line of code.

> “The agent does not receive one giant joined dataset. It begins with a goal and a state. It takes one action, observes evidence, runs a quality gate, and decides whether to continue. When enough evidence exists, it generates an answer. The answer itself is then evaluated and can be retried. Maximum iterations and human review are explicit controls — that is the engineering around the loop.”

### Two loops demonstrated

**Evidence/action loop**  
`Plan → Tool → Observe → Evidence Gate → Re-plan`

**Quality loop**  
`Generate → Judge → Retry / Pass / Human Review`

### Production mapping

The same pattern can later be extended with MLflow Tracing, managed scorers, AI Search, Unity Catalog governance, Lakebase for durable state, and model/agent serving endpoints. They are intentionally left out of the critical execution path here so the webinar demo stays reliable on Free Edition.
